In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.paths import PROCESSED_DATA_DIR,REPORTS_DIR

In [3]:
plt.style.use("default")

In [4]:
X_train, X_test, y_train, y_test = joblib.load(PROCESSED_DATA_DIR / "cell2cell_split.pkl")
df = X_train.copy()
df["Churn"] = y_train.values

In [5]:
print(f"\nDataset Shape : {df.shape}")
print(f"Features      : {df.shape[1]-1}")
print(f"Training Rows : {len(df):,}")


Dataset Shape : (40837, 53)
Features      : 52
Training Rows : 40,837


In [6]:
print("\nTarget Distribution")
print("-" * 40)
print(f"Churn     : {y_train.sum():,} ({y_train.mean():.1%})")
print(f"No Churn  : {(1-y_train).sum():,} ({(1-y_train).mean():.1%})")


Target Distribution
----------------------------------------
Churn     : 11,769 (28.8%)
No Churn  : 29,068 (71.2%)


In [7]:
print(df.dtypes.value_counts())

float64    25
object     20
int64       7
int32       1
Name: count, dtype: int64


In [8]:
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

print(missing[missing > 0].head(20))

HandsetPrice             23214
AgeHH1                     716
AgeHH2                     716
PercChangeMinutes          290
PercChangeRevenues         290
MonthlyMinutes             120
MonthlyRevenue             120
RoamingCalls               120
DirectorAssistedCalls      120
TotalRecurringCharge       120
OverageMinutes             120
ServiceArea                 21
CurrentEquipmentDays         1
HandsetModels                1
Handsets                     1
dtype: int64


In [9]:
print(df.describe().T[[
    "mean",
    "std",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
]].head(15))

                             mean         std     min     25%     50%     75%  \
MonthlyRevenue          58.992476   44.953317     0.0   33.71   48.47   71.03   
MonthlyMinutes         525.242872  528.398044     0.0  158.00  367.00  724.00   
TotalRecurringCharge    46.859002   23.854207   -11.0   30.00   45.00   60.00   
DirectorAssistedCalls    0.897743    2.246531     0.0    0.00    0.25    0.99   
OverageMinutes          40.311344   97.921075     0.0    0.00    3.00   41.00   
RoamingCalls             1.238755   10.406285     0.0    0.00    0.00    0.20   
PercChangeMinutes      -11.750586  257.671706 -3406.0  -83.00   -6.00   65.00   
PercChangeRevenues      -1.119353   41.021682 -1107.7   -7.00   -0.30    1.60   
DroppedCalls             6.005897    8.992924     0.0    0.70    3.00    7.70   
UnansweredCalls         28.294361   38.945442     0.0    5.30   16.30   36.70   
CustomerCareCalls        1.865810    4.961820     0.0    0.00    0.00    1.70   
ThreewayCalls            0.3

In [10]:
fig, ax = plt.subplots(figsize=(6,4))

y_train.value_counts().plot(
    kind="bar",
    color=["steelblue","tomato"],
    ax=ax
)

ax.set_xticklabels(
    ["No Churn (71.2%)","Churn (28.8%)"],
    rotation=0
)

ax.set_title("Target Distribution — Cell2Cell")
ax.set_ylabel("Customers")

plt.tight_layout()

plt.savefig(
    REPORTS_DIR/"eda_target_dist.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

In [11]:
num_cols = df.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

print(f"Numerical Features : {len(num_cols)}")

Numerical Features : 32


In [12]:
skew = (
    df[num_cols]
    .skew()
    .sort_values(key=np.abs, ascending=False)
)

print(skew.head(15))

RoamingCalls                 58.990225
AdjustmentsToCreditRating    19.933861
ThreewayCalls                18.315749
DirectorAssistedCalls        14.674730
CustomerCareCalls            12.877126
CallWaitingCalls             11.384902
PercChangeRevenues            8.731824
RetentionOffersAccepted       8.640305
OverageMinutes                8.361569
RetentionCalls                6.311760
InboundCalls                  5.971880
DroppedBlockedCalls           5.576081
UnansweredCalls               4.476826
DroppedCalls                  4.393481
MonthlyRevenue                4.309883
dtype: float64


In [13]:
corr_with_churn = (
    df[num_cols]
      .corrwith(df["Churn"])
      .abs()
      .sort_values(ascending=False)
)

print(corr_with_churn.head(20))

CurrentEquipmentDays       0.105885
RetentionCalls             0.064419
TotalRecurringCharge       0.062096
MonthlyMinutes             0.048828
UniqueSubs                 0.041152
RetentionOffersAccepted    0.037628
PeakCallsInOut             0.036757
HandsetModels              0.036672
CustomerCareCalls          0.036390
OffPeakCallsInOut          0.035883
PercChangeMinutes          0.035557
ReceivedCalls              0.034594
InboundCalls               0.029482
Handsets                   0.028963
OutboundCalls              0.028512
AgeHH1                     0.027867
UnansweredCalls            0.026584
ThreewayCalls              0.022921
MonthsInService            0.022823
CallWaitingCalls           0.022040
dtype: float64


In [14]:
fig, ax = plt.subplots(figsize=(10,7))

corr_with_churn.head(20).plot(
    kind="barh",
    color="steelblue",
    ax=ax
)

ax.invert_yaxis()

ax.set_title("Top Numerical Correlations with Churn")

plt.tight_layout()

plt.savefig(
    REPORTS_DIR/"eda_corr_bar.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

In [15]:
df["care_calls_bin"] = pd.cut(
    df["CustomerCareCalls"],
    bins=[0,1,2,3,5,10,df["CustomerCareCalls"].max()],
    labels=["0","1","2","3-4","5-9","10+"],
    include_lowest=True
)


In [16]:
care_churn = (
    df.groupby("care_calls_bin")["Churn"]
      .agg(["mean","count"])
)

print(care_churn)

                    mean  count
care_calls_bin                 
0               0.302433  28932
1               0.269508   3191
2               0.245389   1952
3-4             0.254345   2359
5-9             0.254172   2577
10+             0.232749   1826


In [17]:
fig, ax = plt.subplots(figsize=(8,5))

care_churn["mean"].plot(
    kind="bar",
    color="tomato",
    ax=ax
)

ax.set_title("Churn Rate by CustomerCareCalls")

for i,(r,c) in enumerate(zip(
    care_churn["mean"],
    care_churn["count"]
)):
    ax.text(i,r+0.005,f"n={c:,}",ha="center",fontsize=8)

plt.tight_layout()

plt.savefig(
    REPORTS_DIR/"eda_care_calls_churn.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

df.drop(columns="care_calls_bin", inplace=True)

In [18]:
retention = (
    df.groupby("RetentionCalls")["Churn"]
      .agg(["mean","count"])
)

print(retention)

                    mean  count
RetentionCalls                 
0               0.282490  39435
1               0.448916   1292
2               0.432990     97
3               0.454545     11
4               1.000000      2


In [19]:
fig, axes = plt.subplots(1,2,figsize=(14,5))

df[df["Churn"]==0]["CurrentEquipmentDays"].hist(
    bins=50,
    alpha=0.6,
    color="steelblue",
    ax=axes[0],
    label="No Churn"
)

df[df["Churn"]==1]["CurrentEquipmentDays"].hist(
    bins=50,
    alpha=0.6,
    color="tomato",
    ax=axes[0],
    label="Churn"
)

axes[0].legend()

axes[0].set_title(
    "CurrentEquipmentDays Distribution"
)

df["equip_bin"] = pd.cut(
    df["CurrentEquipmentDays"],
    bins=[0,90,180,365,540,730,1000,2000],
    labels=[
        "0-90",
        "90-180",
        "180-365",
        "365-540",
        "540-730",
        "730-1000",
        "1000+"
    ]
)

equip = (
    df.groupby("equip_bin")["Churn"]
      .mean()
)

equip.plot(
    kind="bar",
    color="darkorange",
    ax=axes[1]
)

axes[1].set_title(
    "Churn Rate by Equipment Age"
)

plt.tight_layout()

plt.savefig(
    REPORTS_DIR/"eda_equipment_days.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

df.drop(columns="equip_bin", inplace=True)

In [20]:
ideas = [
    "- High CustomerCareCalls may become complaint_risk feature.",
    "- RetentionCalls may combine with RetentionOffersAccepted.",
    "- Equipment age can be bucketed into loyalty groups.",
    "- Strong numerical predictors should be checked for interactions.",
]

for idea in ideas:
    print(idea)

- High CustomerCareCalls may become complaint_risk feature.
- RetentionCalls may combine with RetentionOffersAccepted.
- Equipment age can be bucketed into loyalty groups.
- Strong numerical predictors should be checked for interactions.


In [21]:
cat_cols = [
    'CreditRating','PrizmCode','Occupation','MaritalStatus',
    'Homeownership','HandsetWebCapable','MadeCallToRetentionTeam',
    'ChildrenInHH','HandsetRefurbished'
]

fig, axes = plt.subplots(3,3,figsize=(18,12))

for ax,col in zip(axes.flatten(),cat_cols):

    churn_rate = (
        df.groupby(col)["Churn"]
        .mean()
        .sort_values(ascending=False)
    )

    churn_rate.plot(
        kind="bar",
        ax=ax,
        color="tomato",
        alpha=0.8
    )

    ax.set_title(f"Churn Rate by {col}")
    ax.set_ylabel("Churn Rate")
    ax.tick_params(axis="x",rotation=45)

plt.tight_layout()
plt.savefig(
    REPORTS_DIR/"eda_categorical_churn.png",
    dpi=150,
    bbox_inches="tight"
)
plt.close()

In [22]:
area_churn = (
    df.groupby("ServiceArea")["Churn"]
    .agg(["mean","count"])
)

print(f"Unique Service Areas : {df['ServiceArea'].nunique()}")
print(area_churn.describe())

fig,ax=plt.subplots(figsize=(10,5))

area_churn["mean"].hist(
    bins=40,
    color="steelblue",
    alpha=0.8,
    ax=ax
)

ax.set_title(
    "Distribution of Area Level Churn Rates"
)

plt.tight_layout()

plt.savefig(
    REPORTS_DIR/"eda_service_area_dist.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

Unique Service Areas : 729
             mean        count
count  729.000000   729.000000
mean     0.283141    55.989026
std      0.215742   124.897749
min      0.000000     1.000000
25%      0.166667     3.000000
50%      0.282609    11.000000
75%      0.362500    50.000000
max      1.000000  1330.000000


In [23]:
for col in cat_cols:

    contingency = pd.crosstab(
        df[col],
        df["Churn"]
    )

    chi2,p,dof,_ = stats.chi2_contingency(
        contingency
    )

    sig = (
        "***" if p<0.001 else
        "**" if p<0.01 else
        "*" if p<0.05 else
        "NS"
    )

    print(
        f"{col:35s}"
        f" chi2={chi2:9.2f}"
        f" p={p:.6f}"
        f" {sig}"
    )

CreditRating                        chi2=   180.90 p=0.000000 ***
PrizmCode                           chi2=    15.56 p=0.001396 **
Occupation                          chi2=     6.11 p=0.527119 NS
MaritalStatus                       chi2=    27.72 p=0.000001 ***
Homeownership                       chi2=     7.73 p=0.005436 **
HandsetWebCapable                   chi2=   152.10 p=0.000000 ***
MadeCallToRetentionTeam             chi2=   181.39 p=0.000000 ***
ChildrenInHH                        chi2=     2.26 p=0.132399 NS
HandsetRefurbished                  chi2=    39.49 p=0.000000 ***


In [24]:
num_cols = df.select_dtypes(
    include=["int64","float64"]
).columns.tolist()

for col in num_cols[:15]:

    valid = df[["Churn",col]].dropna()

    r,p = stats.pointbiserialr(
        valid["Churn"],
        valid[col]
    )

    sig = (
        "***" if p<0.001 else
        "**" if p<0.01 else
        "*" if p<0.05 else
        "NS"
    )

    print(
        f"{col:35s}"
        f" r={r:+.4f}"
        f" p={p:.6f}"
        f" {sig}"
    )

MonthlyRevenue                      r=-0.0104 p=0.036123 *
MonthlyMinutes                      r=-0.0488 p=0.000000 ***
TotalRecurringCharge                r=-0.0621 p=0.000000 ***
DirectorAssistedCalls               r=-0.0155 p=0.001732 **
OverageMinutes                      r=+0.0193 p=0.000094 ***
RoamingCalls                        r=+0.0115 p=0.019793 *
PercChangeMinutes                   r=-0.0356 p=0.000000 ***
PercChangeRevenues                  r=+0.0099 p=0.046663 *
DroppedCalls                        r=-0.0112 p=0.024024 *
UnansweredCalls                     r=-0.0266 p=0.000000 ***
CustomerCareCalls                   r=-0.0364 p=0.000000 ***
ThreewayCalls                       r=-0.0229 p=0.000004 ***
ReceivedCalls                       r=-0.0346 p=0.000000 ***
OutboundCalls                       r=-0.0285 p=0.000000 ***
InboundCalls                        r=-0.0295 p=0.000000 ***


In [25]:
corr = (
    df[num_cols]
    .corrwith(df["Churn"])
    .abs()
    .sort_values(ascending=False)
)

top15 = (
    corr.drop("Churn",errors="ignore")
    .head(15)
    .index
    .tolist()
)

plt.figure(figsize=(14,10))

sns.heatmap(
    df[top15+["Churn"]].corr(),
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    linewidths=0.3
)

plt.title(
    "Correlation Heatmap — Top 15 Features"
)

plt.tight_layout()

plt.savefig(
    REPORTS_DIR/"eda_heatmap.png",
    dpi=150,
    bbox_inches="tight"
)

plt.close()

In [26]:
outlier_cols = [
    "MonthlyRevenue",
    "MonthlyMinutes",
    "CustomerCareCalls",
    "OverageMinutes",
    "TotalRecurringCharge"
]

for col in outlier_cols:

    values = df[col].dropna()

    z = np.abs(stats.zscore(values))

    outliers = (z>3).sum()

    print(
        f"{col:35s}"
        f" max={values.max():8.2f}"
        f" outliers={outliers:6d}"
        f" ({outliers/len(values):.2%})"
    )

print("\nSkewness of Key Variables")
print("-"*50)

print(
    df[outlier_cols]
    .skew()
    .sort_values(
        key=np.abs,
        ascending=False
    )
)

MonthlyRevenue                      max= 1223.38 outliers=   703 (1.73%)
MonthlyMinutes                      max= 7359.00 outliers=   722 (1.77%)
CustomerCareCalls                   max=  327.30 outliers=   721 (1.77%)
OverageMinutes                      max= 4321.00 outliers=   717 (1.76%)
TotalRecurringCharge                max=  400.00 outliers=   428 (1.05%)

Skewness of Key Variables
--------------------------------------------------
CustomerCareCalls       12.877126
OverageMinutes           8.361569
MonthlyRevenue           4.309883
MonthlyMinutes           2.192346
TotalRecurringCharge     1.689620
dtype: float64


In [27]:
conclusions = [
"1. Identify strongest numerical predictors.",
"2. Identify strongest categorical predictors.",
"3. High complaint behaviour deserves engineered features.",
"4. Retention interactions should be engineered.",
"5. Equipment age should be bucketed.",
"6. High-cardinality ServiceArea should use Target Encoding.",
"7. Preserve informative outliers; transform if necessary.",
"8. Use these findings directly in Phase 6."
]

for c in conclusions:
    print(c)


1. Identify strongest numerical predictors.
2. Identify strongest categorical predictors.
3. High complaint behaviour deserves engineered features.
4. Retention interactions should be engineered.
5. Equipment age should be bucketed.
6. High-cardinality ServiceArea should use Target Encoding.
7. Preserve informative outliers; transform if necessary.
8. Use these findings directly in Phase 6.


In [28]:
ideas = [
"complaint_risk",
"retention_failed",
"escalated_customer",
"old_equipment",
"equipment_age_bucket",
"premium_dissatisfied",
"servicearea_target_encoding",
"log_customer_care_calls"
]

for idea in ideas:
    print("•",idea)

• complaint_risk
• retention_failed
• escalated_customer
• old_equipment
• equipment_age_bucket
• premium_dissatisfied
• servicearea_target_encoding
• log_customer_care_calls
